# Forecasting Optuna Search CV 
## Modulus Set 1

**Notebook Goal**
- A modeling pipeline that optimizes the hyperparameters of the sktime forecasters that have the [capavility:pred_int tag](https://www.sktime.net/en/stable/examples/01b_forecasting_proba.html) 
- This notebook will focus on the ones where `i mod 4 = 1` wher `i` is the index of the registry table in the above link.
- The work will be based on this documentation: [ForecastingOptunaSearchCV](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.forecasting.model_selection.ForecastingOptunaSearchCV.html)

In [1]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

In [2]:
from sktime.registry import all_estimators

forecasters = all_estimators(
    "forecaster", filter_tags={"capability:pred_int": True}, as_dataframe=True
)

# Get all selected forecasters (mod set 1)
selected_forecasters = forecasters[forecasters.index % 4 == 1] # i mod 4 == 1
selected_forecasters

/Users/alyss/UW/blood-glucose-control/nocturnal-hypo-gly-prob-forecast/env/lib/python3.12/site-packages/gluonts/json.py:102: UserWarning: Using `json`-module for json-handling. Consider installing one of `orjson`, `ujson` to speed up serialization and deserialization.
  warnings.warn(


,name,object
1,ARIMA,<class 'sktime.forecasting.arima._pmdarima.ARI...
5,BaggingForecaster,<class 'sktime.forecasting.compose._bagging.Ba...
9,DartsLinearRegressionModel,<class 'sktime.forecasting.darts.DartsLinearRe...
13,DirectTabularRegressionForecaster,<class 'sktime.forecasting.compose._reduce.Dir...
17,FhPlexForecaster,<class 'sktime.forecasting.compose._fhplex.FhP...
21,ForecastingPipeline,<class 'sktime.forecasting.compose._pipeline.F...
25,MultioutputTabularRegressionForecaster,<class 'sktime.forecasting.compose._reduce.Mul...
29,Permute,<class 'sktime.forecasting.compose._pipeline.P...
33,Prophetverse,<class 'sktime.forecasting.prophetverse.Prophe...
37,RecursiveTabularRegressionForecaster,<class 'sktime.forecasting.compose._reduce.Rec...


### Optimizing ARIMA hyperparameters

In [3]:
forecaster_class = selected_forecasters.iloc[0, 1] # Get the first forecaster to optimize
forecaster = forecaster_class()

print(forecaster_class.__name__)

forecaster.get_params()

ARIMA


{'concentrate_scale': False,
 'enforce_invertibility': True,
 'enforce_stationarity': True,
 'hamilton_representation': False,
 'maxiter': 50,
 'measurement_error': False,
 'method': 'lbfgs',
 'mle_regression': True,
 'order': (1, 0, 0),
 'out_of_sample_size': 0,
 'scoring': 'mse',
 'scoring_args': None,
 'seasonal_order': (0, 0, 0, 0),
 'simple_differencing': False,
 'start_params': None,
 'suppress_warnings': False,
 'time_varying_regression': False,
 'trend': None,
 'with_intercept': True}

In [12]:
from sktime.forecasting.model_selection import ForecastingOptunaSearchCV
from sktime.forecasting.model_selection import ExpandingWindowSplitter
from sktime.datasets import load_shampoo_sales
from sktime.split import temporal_train_test_split
from sktime.forecasting.base import ForecastingHorizon
from optuna.distributions import CategoricalDistribution, IntDistribution

# Load data and split into training and test sets
y = load_shampoo_sales()
y_train, y_test = temporal_train_test_split(y=y, test_size=6)

# Define forecasting horizon
fh = ForecastingHorizon(y_test.index, is_relative=False).to_relative(cutoff=y_train.index[-1])

# Cross-validation strategy
cv = ExpandingWindowSplitter(fh=fh, initial_window=24, step_length=1)

# Define parameter grid for ARIMA (focusing on the relevant model parameters)
param_grid = {
	'measurement_error': CategoricalDistribution([True, False]),
	'method': CategoricalDistribution([
        "newton", "nm", "bfgs", "lbfgs", "powell", 
        "cg", "ncg", "basinhopping"
    ]),
	'mle_regression': CategoricalDistribution([True, False]),
	'order': CategoricalDistribution([(1, 0, 0), (1, 1, 0)]),
	'out_of_sample_size': IntDistribution(0, 3),
	'scoring': CategoricalDistribution([
        "mse", "mae", "smape", "mape", "mase", "rmse"
    ]),
	'time_varying_regression': CategoricalDistribution([True, False]),
	'trend': CategoricalDistribution(["n", "c", "t", "ct"]),
	'with_intercept': CategoricalDistribution([True, False]),
}

# Optimize hyperparameters
gscv = ForecastingOptunaSearchCV(
    forecaster=forecaster,
    param_grid=param_grid,
    cv=cv,
    n_evals=10,  # Number of trials
)

# Fit and find best params
gscv.fit(y)
gscv.best_params_

/Users/alyss/UW/blood-glucose-control/nocturnal-hypo-gly-prob-forecast/env/lib/python3.12/site-packages/optuna/distributions.py:524: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 0, 0) which is of type tuple.
  warnings.warn(message)
/Users/alyss/UW/blood-glucose-control/nocturnal-hypo-gly-prob-forecast/env/lib/python3.12/site-packages/optuna/distributions.py:524: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1, 0) which is of type tuple.
  warnings.warn(message)
/Users/alyss/UW/blood-glucose-control/nocturnal-hypo-gly-prob-forecast/env/lib/python3.12/site-packages/sktime/forecasting/model_selection/_tune.py:1773: UserWarning: ForecastingOptunaSearchCV is experimental, and interfaces may change. User feedback and suggestions for future development are appreciated in issue #6618 here: https://g

{'measurement_error': True,
 'method': 'lbfgs',
 'mle_regression': False,
 'order': (1, 1, 0),
 'out_of_sample_size': 2,
 'scoring': 'mae',
 'time_varying_regression': True,
 'trend': 'c',
 'with_intercept': True}